# Ejercicio 10: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [2]:
!pip install -q beir rank_bm25 sentence-transformers torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 16.4 MB/s eta 0:00:00


In [3]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

/usr/local/lib/python3.12/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [4]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

../data/beir_datasets/scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

'../data/beir_datasets/scifact'

In [5]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

  0%|          | 0/5183 [00:00<?, ?it/s]

In [6]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [7]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [8]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [9]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [10]:
from rank_bm25 import BM25Okapi
import re

def tokenize(text):
    return re.findall(r"\w+", text.lower())

corpus_texts = [
    tokenize(doc["title"] + " " + doc["text"])
    for doc in corpus.values()
]

bm25 = BM25Okapi(corpus_texts)
doc_ids = list(corpus.keys())

In [11]:
def bm25_retrieve(query, k=10):
    tokenized_query = tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    
    ranked = sorted(
        zip(doc_ids, scores),
        key=lambda x: x[1],
        reverse=True
    )
    
    return ranked[:k]

In [12]:
bm25_results = bm25_retrieve(
    df_queries.loc[df_queries["query_id"] == qid, "query"].values[0],
    k=10
)

bm25_results

[('5270265', np.float64(54.37032055605303)),
 ('26688294', np.float64(53.98278624415536)),
 ('19752008', np.float64(53.68065228794002)),
 ('45764440', np.float64(52.90211244897466)),
 ('16280642', np.float64(52.48082747638017)),
 ('12785130', np.float64(51.38924361223814)),
 ('5914739', np.float64(50.40944635803289)),
 ('11200685', np.float64(49.13409314866078)),
 ('37964706', np.float64(48.74205432944344)),
 ('35660758', np.float64(48.502791869979625))]

In [13]:
# Evaluación BM25 (Recall y nDGC)
from beir.retrieval.evaluation import EvaluateRetrieval

retriever = EvaluateRetrieval(None, k_values=[10])

bm25_run = {}
for qid_, query in queries.items():
    results = bm25_retrieve(query, k=10)
    bm25_run[qid_] = {doc_id: score for doc_id, score in results}

bm25_metrics = retriever.evaluate(qrels, bm25_run, retriever.k_values)
bm25_metrics

({'NDCG@10': 0.65189},
 {'MAP@10': 0.60697},
 {'Recall@10': 0.774},
 {'P@10': 0.085})

## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [14]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

2026-01-13 03:37:25.327436: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768275445.589940      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768275445.672237      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768275446.286799      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768275446.286861      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768275446.286865      55 computation_placer.cc:177] computation placer alr

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [15]:
def cross_encoder_rerank(query, candidates):
    pairs = [(query, corpus[doc_id]["text"]) for doc_id, _ in candidates]
    scores = cross_encoder.predict(pairs)
    
    reranked = sorted(
        zip([doc_id for doc_id, _ in candidates], scores),
        key=lambda x: x[1],
        reverse=True
    )
    return reranked

In [16]:
reranked = cross_encoder_rerank(
    df_queries.loc[df_queries["query_id"] == qid, "query"].values[0],
    bm25_results
)

pd.DataFrame({
    "BM25": [doc for doc, _ in bm25_results],
    "CrossEncoder": [doc for doc, _ in reranked]
})

,BM25,CrossEncoder
0,5270265,16280642
1,26688294,35660758
2,19752008,19752008
3,45764440,5914739
4,16280642,37964706
5,12785130,11200685
6,5914739,45764440
7,11200685,12785130
8,37964706,26688294
9,35660758,5270265


## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [17]:
import numpy as np

def build_features(query, doc_id, bm25_score):
    return [
        bm25_score,
        len(corpus[doc_id]["text"]),
        len(query)
    ]

In [18]:
from sklearn.ensemble import RandomForestRegressor

X, y = [], []

for _, row in df_qrels.iterrows():
    q = queries[row["query_id"]]
    d = row["doc_id"]
    bm25_score = dict(bm25_run[row["query_id"]]).get(d, 0)
    
    X.append(build_features(q, d, bm25_score))
    y.append(row["relevance"])

ltr_model = RandomForestRegressor(n_estimators=100)
ltr_model.fit(X, y)

RandomForestRegressor()

In [19]:
def ltr_rerank(query, candidates):
    feats = [
        build_features(query, doc_id, score)
        for doc_id, score in candidates
    ]
    scores = ltr_model.predict(feats)
    
    return sorted(
        zip([doc_id for doc_id, _ in candidates], scores),
        key=lambda x: x[1],
        reverse=True
    )

## Parte 5. Evaluación post re-ranking

Calcular métricas:
* nDCG@10
* MAP
* Recall@10

In [21]:
reranked_run = {}

for qid_, query in queries.items():
    candidates = bm25_retrieve(query, k=10)
    reranked_docs = cross_encoder_rerank(query, candidates)

    # ⚠️ Filtrar solo docs que aparecen en qrels
    valid_docs = qrels.get(qid_, {})
    
    reranked_run[qid_] = {
        doc_id: float(score)
        for doc_id, score in reranked_docs
        if doc_id in valid_docs
    }

In [22]:
retriever.evaluate(qrels, reranked_run, retriever.k_values)

({'NDCG@10': 0.77917},
 {'MAP@10': 0.774},
 {'Recall@10': 0.774},
 {'P@10': 0.085})